## DIM_PARTICULAR
Distinct category–particular pairs from regional appropriations and obligations.

In [0]:
DROP TABLE IF EXISTS pondolens.dev_gold.dim_particular

In [0]:
%sql
CREATE TABLE IF NOT EXISTS pondolens.dev_gold.dim_particular
USING DELTA
AS
WITH distinct_particulars AS (
  SELECT
    UPPER(TRIM(CAST(category AS STRING))) AS category,
    UPPER(TRIM(CAST(particular AS STRING))) AS particular
  FROM pondolens.dev_silver.regional_appropriations
  WHERE NULLIF(TRIM(CAST(category AS STRING)), '') IS NOT NULL
    AND NULLIF(TRIM(CAST(particular AS STRING)), '') IS NOT NULL
    AND UPPER(TRIM(CAST(particular AS STRING))) <> 'ALL'

  UNION

  SELECT
    UPPER(TRIM(CAST(category AS STRING))) AS category,
    UPPER(TRIM(CAST(particular AS STRING))) AS particular
  FROM pondolens.dev_silver.regional_obligations
  WHERE NULLIF(TRIM(CAST(category AS STRING)), '') IS NOT NULL
    AND NULLIF(TRIM(CAST(particular AS STRING)), '') IS NOT NULL
    AND UPPER(TRIM(CAST(particular AS STRING))) <> 'ALL'
),
id_start AS (
  SELECT 29999 AS starting_id
)
SELECT
  CAST(
    b.starting_id
      + ROW_NUMBER() OVER (ORDER BY p.category, p.particular)
    AS INT
  ) AS particular_id,
  CAST(p.category AS STRING) AS category,
  CAST(p.particular AS STRING) AS particular
FROM distinct_particulars AS p
CROSS JOIN id_start AS b;

### RESULT

In [0]:
%sql
SELECT *
FROM pondolens.dev_gold.dim_particular

### Data Quality Check

In [0]:
%sql
WITH settings AS (
  -- Thresholds apply only to missing source pairs.
  SELECT
    5 AS pass_max,
    10 AS review_max
),
source_pairs AS (
  SELECT
    UPPER(TRIM(CAST(category AS STRING))) AS category,
    UPPER(TRIM(CAST(particular AS STRING))) AS particular
  FROM pondolens.dev_silver.regional_appropriations
  WHERE NULLIF(TRIM(CAST(category AS STRING)), '') IS NOT NULL
    AND NULLIF(TRIM(CAST(particular AS STRING)), '') IS NOT NULL
    AND UPPER(TRIM(CAST(particular AS STRING))) <> 'ALL'

  UNION

  SELECT
    UPPER(TRIM(CAST(category AS STRING))) AS category,
    UPPER(TRIM(CAST(particular AS STRING))) AS particular
  FROM pondolens.dev_silver.regional_obligations
  WHERE NULLIF(TRIM(CAST(category AS STRING)), '') IS NOT NULL
    AND NULLIF(TRIM(CAST(particular AS STRING)), '') IS NOT NULL
    AND UPPER(TRIM(CAST(particular AS STRING))) <> 'ALL'
),
dim AS (
  SELECT *
  FROM pondolens.dev_gold.dim_particular
),
totals AS (
  SELECT
    (SELECT COUNT(*) FROM dim) AS dim_rows,
    (SELECT COUNT(*) FROM source_pairs) AS source_pair_count
),
duplicate_ids AS (
  SELECT particular_id
  FROM dim
  WHERE particular_id IS NOT NULL
  GROUP BY particular_id
  HAVING COUNT(*) > 1
),
duplicate_pairs AS (
  SELECT
    UPPER(TRIM(category)) AS category,
    UPPER(TRIM(particular)) AS particular
  FROM dim
  GROUP BY UPPER(TRIM(category)), UPPER(TRIM(particular))
  HAVING COUNT(*) > 1
),
checks AS (
  SELECT
    'Missing IDs or blank category/particular' AS dq_check,
    COUNT(*) AS affected_count,
    'dimension rows' AS count_basis
  FROM dim
  WHERE particular_id IS NULL
    OR NULLIF(TRIM(category), '') IS NULL
    OR NULLIF(TRIM(particular), '') IS NULL

  UNION ALL

  -- Count every row belonging to a duplicate group.
  SELECT
    'Duplicate particular IDs',
    COUNT(*),
    'dimension rows'
  FROM dim AS d
  LEFT SEMI JOIN duplicate_ids AS x
    ON d.particular_id = x.particular_id

  UNION ALL

  SELECT
    'Duplicate normalized category-particular pairs',
    COUNT(*),
    'dimension rows'
  FROM dim AS d
  LEFT SEMI JOIN duplicate_pairs AS x
    ON UPPER(TRIM(d.category)) <=> x.category
    AND UPPER(TRIM(d.particular)) <=> x.particular

  UNION ALL

  SELECT
    'IDs below 30000',
    COUNT(*),
    'dimension rows'
  FROM dim
  WHERE particular_id < 30000

  UNION ALL

  SELECT
    'Text not uppercase or trimmed',
    COUNT(*),
    'dimension rows'
  FROM dim
  WHERE category <> UPPER(TRIM(category))
     OR particular <> UPPER(TRIM(particular))

  UNION ALL

  SELECT
    'Excluded ALL rows present',
    COUNT(*),
    'dimension rows'
  FROM dim
  WHERE UPPER(TRIM(particular)) = 'ALL'

  UNION ALL

  SELECT
    'Source pairs missing from dimension',
    COUNT(*),
    'source pairs'
  FROM source_pairs AS s
  LEFT ANTI JOIN dim AS d
    ON s.category = d.category
    AND s.particular = d.particular

  UNION ALL

  SELECT
    'Incorrect or missing column types',
    3 - COUNT(*),
    'columns'
  FROM pondolens.information_schema.columns
  WHERE table_schema = 'dev_gold'
    AND table_name = 'dim_particular'
    AND (
      (column_name = 'particular_id' AND UPPER(data_type) = 'INT')
      OR
      (column_name IN ('category', 'particular')
        AND UPPER(data_type) = 'STRING')
    )
),
evaluated AS (
  SELECT
    c.*,
    CASE c.count_basis
      WHEN 'source pairs' THEN t.source_pair_count
      WHEN 'columns' THEN 3
      ELSE t.dim_rows
    END AS total_checked,
    CASE
      WHEN c.dq_check = 'Source pairs missing from dimension'
        THEN s.pass_max
      ELSE 0
    END AS pass_max,
    CASE
      WHEN c.dq_check = 'Source pairs missing from dimension'
        THEN s.review_max
      ELSE 0
    END AS review_max
  FROM checks AS c
  CROSS JOIN totals AS t
  CROSS JOIN settings AS s
)
SELECT
  dq_check,
  affected_count,
  count_basis,
  total_checked,
  ROUND(
    100.0 * affected_count / NULLIF(total_checked, 0),
    2
  ) AS affected_percentage,
  pass_max,
  review_max,
  CASE
    WHEN total_checked = 0 THEN 'REVIEW'
    WHEN affected_count <= pass_max THEN 'PASS'
    WHEN affected_count <= review_max THEN 'REVIEW'
    ELSE 'FAIL'
  END AS status
FROM evaluated
ORDER BY dq_check;